# TRACTOR Inference Notebook
Load models with different T, sample input from a raw KPM CSV, and run inference.

No care of accuracy.

Latency we measured from TRACTOR was 2 to 7.7 ms on normal Ubuntu server.

TRACTOR paper: https://ieeexplore.ieee.org/abstract/document/10622798

## 0 — Config

In [22]:
import sys

MODEL_DIR  = "/content"
CSV_PATH   = "/content/urllc1010123456002_metrics.csv"
NORM_PATH  = f"{MODEL_DIR}/cols_maxmin.pkl"

MODEL_TYPE = "CNN"
T_VALUES   = [32] # window sizes to test, i.e., 4, 8, 16, 32 -> importantly, this changes the input dimension
NCLASS     = 4
N_SAMPLE_WINDOWS = 200      # how many windows to run inference on per T, no need to care this. Simply, this example makes 200 predictions.

CLASS_NAMES = {0: "eMBB", 1: "mMTC", 2: "URLLC", 3: "Ctrl"}
MODEL_FILE  = {"CNN": "model.{T}.cnn.pt"}

## 1 — Load normalizer

In [23]:
import pickle
import numpy as np

# Features to exclude from the 31-column array (matches exclude_param in visual_xapp_inference.py)
EXCLUDE = [0, 1, 2, 3, 4, 5, 6, 7, 8, 14, 22, 27, 28, 29]
INDEXES_TO_KEEP = np.array([i for i in range(31) if i not in EXCLUDE])
# Expected: [ 9 10 11 12 13 15 16 17 18 19 20 21 23 24 25 26 30 ]

colsparams = pickle.load(open(NORM_PATH, "rb"))
K = len(colsparams)  # number of features after filtering

print(f"Indexes to keep ({len(INDEXES_TO_KEEP)}): {INDEXES_TO_KEEP}")
print(f"K (features after filtering): {K}")
print("\nNorm ranges (feature index → min, max):")
for f in sorted(colsparams):
    print(f"  [{f:2d}]  min={colsparams[f]['min']:.4f}  max={colsparams[f]['max']:.4f}")

Indexes to keep (17): [ 9 10 11 12 13 15 16 17 18 19 20 21 23 24 25 26 30]
K (features after filtering): 17

Norm ranges (feature index → min, max):
  [ 0]  min=0.0000  max=27.7311
  [ 1]  min=0.0000  max=879.0000
  [ 2]  min=0.0000  max=193816.0000
  [ 3]  min=0.0000  max=7.6869
  [ 4]  min=0.0000  max=264.0000
  [ 5]  min=0.1000  max=15.0000
  [ 6]  min=0.0000  max=16.3000
  [ 7]  min=0.0000  max=875.0000
  [ 8]  min=0.0000  max=150000.0000
  [ 9]  min=0.0000  max=14.9592
  [10]  min=0.0000  max=250.0000
  [11]  min=0.0000  max=100.0000
  [12]  min=0.0000  max=39.8865
  [13]  min=0.0000  max=31.0000
  [14]  min=0.0000  max=13200.0000
  [15]  min=0.0000  max=4518.0000
  [16]  min=0.0000  max=10.0000


## 2 — Load and preview CSV

In [24]:
import pandas as pd
import os

df_raw = pd.read_csv(CSV_PATH)
print(f"Raw shape: {df_raw.shape}")

# Drop the 5 unnamed separator columns — leaves exactly 31 columns
df = df_raw.drop(columns=[c for c in df_raw.columns if c.startswith("Unnamed")])
print(f"After dropping unnamed cols: {df.shape}")
print(f"Columns ({len(df.columns)}): {list(df.columns)}")
df.head(3)

Raw shape: (19231, 36)
After dropping unnamed cols: (19231, 31)
Columns (31): ['Timestamp', 'num_ues', 'IMSI', 'RNTI', 'slicing_enabled', 'slice_id', 'slice_prb', 'power_multiplier', 'scheduling_policy', 'dl_mcs', 'dl_n_samples', 'dl_buffer [bytes]', 'tx_brate downlink [Mbps]', 'tx_pkts downlink', 'tx_errors downlink (%)', 'dl_cqi', 'ul_mcs', 'ul_n_samples', 'ul_buffer [bytes]', 'rx_brate uplink [Mbps]', 'rx_pkts uplink', 'rx_errors uplink (%)', 'ul_rssi', 'ul_sinr', 'phr', 'sum_requested_prbs', 'sum_granted_prbs', 'dl_pmi', 'dl_ri', 'ul_n', 'ul_turbo_iters']


,Timestamp,num_ues,IMSI,RNTI,slicing_enabled,slice_id,slice_prb,power_multiplier,scheduling_policy,dl_mcs,...,rx_errors uplink (%),ul_rssi,ul_sinr,phr,sum_requested_prbs,sum_granted_prbs,dl_pmi,dl_ri,ul_n,ul_turbo_iters
0,1667166389925,1,1010123456002,70,1,1,18,1,1,2.425,...,0.0,0,35.7610,31,1,39,0,0,0,1.0
1,1667166390175,1,1010123456002,70,1,1,18,1,1,0.000,...,0.0,0,35.8251,31,0,3,0,0,0,1.0
2,1667166390425,1,1010123456002,70,1,1,18,1,1,0.000,...,0.0,0,0.0000,0,0,0,0,0,0,0.0


In [25]:
# Derive ground-truth label from filename
fname = os.path.basename(CSV_PATH).lower()
if   "embb" in fname: TRUE_LABEL = 0
elif "mmtc" in fname: TRUE_LABEL = 1
elif "urll" in fname: TRUE_LABEL = 2
else:                 TRUE_LABEL = 3
print(f"Ground-truth label from filename: {TRUE_LABEL} ({CLASS_NAMES[TRUE_LABEL]})")

Ground-truth label from filename: 2 (URLLC)


## 3 — Normalize

In [26]:
def normalize_window(window_TxN, indexes_to_keep, colsparams):
    """Filter to K features then min-max normalize. Returns (T, K) array."""
    w = window_TxN[:, indexes_to_keep].astype(float)  # (T, K)
    for f in range(w.shape[1]):
        mn = colsparams[f]['min']
        mx = colsparams[f]['max']
        w[:, f] = np.clip(w[:, f], mn, mx)
        denom = (mx - mn) if mx != mn else 1.0
        w[:, f] = (w[:, f] - mn) / denom
    return w

# Quick sanity check on a single row
raw_31 = df.to_numpy()  # shape (N, 31)
sample_window = raw_31[:4]  # T=4 window
norm_window = normalize_window(sample_window, INDEXES_TO_KEEP, colsparams)
print(f"Input shape (raw):        {sample_window.shape}")
print(f"Input shape (normalized): {norm_window.shape}")
print(f"Value range after norm:   [{norm_window.min():.3f}, {norm_window.max():.3f}]")

Input shape (raw):        (4, 31)
Input shape (normalized): (4, 17)
Value range after norm:   [0.000, 1.000]


In [27]:
# Side-by-side comparison: raw vs normalised for the 17 kept features
kept_names = df.columns[INDEXES_TO_KEEP].tolist()
comparison = pd.DataFrame({
    "feature":   kept_names,
    "raw_row0":  sample_window[0, INDEXES_TO_KEEP],
    "norm_row0": norm_window[0],
})
comparison

,feature,raw_row0,norm_row0
0,dl_mcs,2.425000,0.087447
1,dl_n_samples,40.000000,0.045506
2,dl_buffer [bytes],0.000000,0.000000
3,tx_brate downlink [Mbps],0.009152,0.001191
4,tx_pkts downlink,10.000000,0.037879
5,dl_cqi,9.000000,0.597315
6,ul_mcs,11.622200,0.713018
7,ul_n_samples,45.000000,0.051429
8,ul_buffer [bytes],0.000000,0.000000
9,rx_brate uplink [Mbps],0.129984,0.008689


## 4 — Load models for different T

In [28]:
import torch
from torch import nn
from torch.optim.lr_scheduler import ReduceLROnPlateau
import numpy as np
import math

# Define model
class ConvNN(nn.Module):
    def __init__(self, numChannels=1, slice_len=4, num_feats=17, classes=4):
        super(ConvNN, self).__init__()

        self.numChannels = numChannels

        # initialize first set of CONV => RELU => POOL layers
        self.conv1 = nn.Conv2d(in_channels=numChannels, out_channels=20,
                               kernel_size=(4, 1))
        self.relu1 = nn.ReLU()
        # self.maxpool1 = nn.MaxPool2d(kernel_size=(2, 1), stride=(2, 2))
        ##  initialize second set of CONV => RELU => POOL layers
        # self.conv2 = nn.Conv2d(in_channels=20, out_channels=50,
        #                    kernel_size=(5, 5))
        # self.relu2 = nn.ReLU()
        # self.maxpool2 = nn.MaxPool2d(kernel_size=(2, 2), stride=(2, 2))
        ## initialize first (and only) set of FC => RELU layers

        # pass a random input
        rand_x = torch.Tensor(np.random.random((1, 1, slice_len, num_feats)))
        output_size = torch.flatten(self.conv1(rand_x)).shape
        self.fc1 = nn.Linear(in_features=output_size.numel(), out_features=512)
        self.relu3 = nn.ReLU()
        # initialize our softmax classifier
        self.fc2 = nn.Linear(in_features=512, out_features=classes)
        self.logSoftmax = nn.LogSoftmax(dim=1)

    def forward(self, x):
        x = x.reshape \
            ((x.shape[0], self.numChannels, x.shape[1], x.shape[2]))   # CNN 2D expects a [N, Cin, H, W] size of data
        # pass the input through our first set of CONV => RELU =>
        # POOL layers
        x = self.conv1(x)
        x = self.relu1(x)
        # x = self.maxpool1(x)
        ## pass the output from the previous layer through the second
        ## set of CONV => RELU => POOL layers
        # x = self.conv2(x)
        # x = self.relu2(x)
        # x = self.maxpool2(x)
        ## flatten the output from the previous layer and pass it
        ## through our only set of FC => RELU layers
        x = torch.flatten(x, 1)
        x = self.fc1(x)
        x = self.relu3(x)
        # pass the output to our softmax classifier to get our output
        # predictions
        x = self.fc2(x)
        output = self.logSoftmax(x)
        # return the output predictions
        return output

In [29]:
import torch

def load_model(model_type, T, K, nclass, model_dir):
    filename = MODEL_FILE[model_type].format(T=T)
    path = f"{model_dir}/{filename}"
    if model_type == "CNN":
        m = ConvNN(classes=nclass, slice_len=T, num_feats=K)

    device = "cuda" if torch.cuda.is_available() else "cpu"
    state = torch.load(path, map_location=device)
    m.load_state_dict(state["model_state_dict"])
    m.to(device)
    m.eval()
    n_params = sum(p.numel() for p in m.parameters())

    return m, device, n_params

models = {}
for T in T_VALUES:
    m, dev, n = load_model(MODEL_TYPE, T, K, NCLASS, MODEL_DIR)
    models[T] = {"model": m, "device": dev}
    print(f"T={T:2d} | {MODEL_TYPE} loaded on {dev} | params: {n:,}")

T=32 | CNN loaded on cpu | params: 5,050,984


## 5 — Build windows and run inference

In [30]:
from collections import deque

results = []  # rows: {T, window_id, predicted_class, class_name, correct}

for T in T_VALUES:
    m   = models[T]["model"]
    dev = models[T]["device"]
    buf = deque(maxlen=T)
    win_id = 0

    for row in raw_31:
        # Normalize single row (1, K)
        norm_row = normalize_window(row[np.newaxis, :], INDEXES_TO_KEEP, colsparams)[0]
        buf.append(norm_row)

        if len(buf) == T:  # window is full
            window = np.array(buf)          # (T, K)
            t = torch.tensor(window[np.newaxis], dtype=torch.float32).to(dev)  # (1, T, K)
            with torch.no_grad():
                logits = m(t)
            pred = int(logits.argmax(1).item())
            results.append({"T": T, "window_id": win_id, "pred": pred,
                            "class_name": CLASS_NAMES[pred], "correct": pred == TRUE_LABEL})
            win_id += 1
            if win_id >= N_SAMPLE_WINDOWS:
                break

df_results = pd.DataFrame(results)
print(df_results.head(12).to_string(index=False))

 T  window_id  pred class_name  correct
32          0     0       eMBB    False
32          1     0       eMBB    False
32          2     0       eMBB    False
32          3     0       eMBB    False
32          4     0       eMBB    False
32          5     0       eMBB    False
32          6     0       eMBB    False
32          7     0       eMBB    False
32          8     0       eMBB    False
32          9     0       eMBB    False
32         10     0       eMBB    False
32         11     0       eMBB    False


In [31]:
# Accuracy per T
summary = (
    df_results.groupby("T")
    .agg(accuracy=("correct", "mean"), n_windows=("window_id", "count"))
    .assign(accuracy=lambda x: (x["accuracy"] * 100).round(1))
    .reset_index()
)
summary.columns = ["T", "Accuracy (%)", "Windows"]
print(f"Ground truth: {TRUE_LABEL} ({CLASS_NAMES[TRUE_LABEL]})\n")
print(summary.to_string(index=False))

Ground truth: 2 (URLLC)

 T  Accuracy (%)  Windows
32           0.0      200
